In [13]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
from collections import Counter
from sklearn.metrics import roc_auc_score, accuracy_score, log_loss
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import warnings
warnings.filterwarnings('ignore')

In [16]:
DATA_PATH=r"D:\GitRepo\student-performance-forecast\data\processed\test"

In [17]:
files = [f for f in os.listdir(DATA_PATH) if f.endswith('.parquet')]
print(f"Найдено файлов: {len(files)}")

files_sample = files[:2000]
print(f"Используем {len(files_sample)} файлов")

print("\n=== Анализ категориальных признаков ===")

all_sources = set()
source_counts = Counter()
platform_counts = Counter()

for file in tqdm(files_sample, desc="Анализ"):
    df = pd.read_parquet(os.path.join(DATA_PATH, file))
    if 'source' in df.columns:
        for src in df['source'].unique():
            all_sources.add(src)
            source_counts[src] += 1
    if 'platform' in df.columns:
        for plat in df['platform'].unique():
            platform_counts[plat] += 1

top_sources = [src for src, _ in source_counts.most_common(15)]
source_to_idx = {src: i for i, src in enumerate(top_sources)}
source_to_idx['other'] = len(top_sources)

platform_to_idx = {'mobile': 1, 'web': 0}
for plat in platform_counts.keys():
    if plat not in platform_to_idx:
        platform_to_idx[plat] = 2

print(f"Source классов: {len(source_to_idx)}")
print(f"Platform классов: {len(platform_to_idx)}")

def create_sequences(df, source_to_idx, platform_to_idx, seq_len=10):
    """
    Создает последовательности для предсказания следующего ответа
    """
    df = df.copy()
    df = df.sort_values('timestamp').reset_index(drop=True)
    
    if len(df) < seq_len + 1:
        return None, None, None
    
    # Нормализация числовых
    df['time_norm'] = np.clip(df['time_to_answer_ms'] / 30000, 0, 1)
    df['attempt_norm'] = np.clip(df['attempt_count'] / 5, 0, 1)
    df['lectures_norm'] = np.clip(df['lectures_between_bundles'] / 10, 0, 1)
    
    # Кодирование категориальных
    df['platform_code'] = df['platform'].map(platform_to_idx).fillna(2).astype('int8')
    df['source_code'] = df['source'].map(source_to_idx).fillna(source_to_idx['other']).astype('int16')
    
    # Временные признаки
    df['hour'] = pd.to_datetime(df['timestamp'], unit='ms').dt.hour / 23
    df['day_of_week'] = pd.to_datetime(df['timestamp'], unit='ms').dt.dayofweek / 6
    
    # Исторические признаки
    # Скользящая средняя успешности за последние N вопросов
    df['rolling_accuracy'] = df['correct_flag'].rolling(window=seq_len, min_periods=1).mean()
    
    # Время с последнего действия (в минутах)
    df['time_since_last'] = df['timestamp'].diff().fillna(0) / 60000
    df['time_since_last'] = np.clip(df['time_since_last'] / 60, 0, 1)  # нормализация до 1 часа
    
    # Создание признакового вектора
    # Каждое действие представлено 11 признаками
    features = np.column_stack([
        df['attempt_norm'].values,           # 0: количество попыток
        df['time_norm'].values,              # 1: время ответа
        df['lectures_norm'].values,          # 2: просмотр лекций
        df['platform_code'].values,          # 3: платформа
        df['source_code'].values,            # 4: источник
        df['hour'].values,                   # 5: час дня
        df['day_of_week'].values,            # 6: день недели
        df['rolling_accuracy'].values,       # 7: скользящая успешность
        df['time_since_last'].values,        # 8: время с последнего действия
        df['correct_flag'].values,           # 9: правильность текущего ответа
        df['correct_flag'].shift(1).fillna(0).values  # 10: предыдущий ответ
    ]).astype(np.float32)
    
    # Создание последовательностей
    X, y = [], []
    for i in range(seq_len, len(features)):
        X.append(features[i-seq_len:i])
        y.append(features[i, 9])  # правильность следующего ответа
    
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32), df

# Загрузка данных
print("\n=== Загрузка и создание последовательностей ===")

all_X, all_y = [], []

for file in tqdm(files_sample, desc="Обработка"):
    df = pd.read_parquet(os.path.join(DATA_PATH, file))
    
    X, y, _ = create_sequences(df, source_to_idx, platform_to_idx, seq_len=10)
    
    if X is not None and len(X) > 0:
        all_X.append(X)
        all_y.append(y)

X = np.concatenate(all_X, axis=0)
y = np.concatenate(all_y, axis=0)

print(f"\nX shape: {X.shape} (примеров, временных шагов, признаков)")
print(f"y shape: {y.shape}")
print(f"Правильных ответов: {y.sum():.0f}/{len(y)} ({y.mean()*100:.1f}%)")


Найдено файлов: 1000
Используем 1000 файлов

=== Анализ категориальных признаков ===


Анализ: 100%|██████████| 1000/1000 [00:02<00:00, 423.01it/s]


Source классов: 8
Platform классов: 2

=== Загрузка и создание последовательностей ===


Обработка: 100%|██████████| 1000/1000 [00:12<00:00, 79.38it/s]


X shape: (282467, 10, 11) (примеров, временных шагов, признаков)
y shape: (282467,)
Правильных ответов: 196994/282467 (69.7%)


In [22]:
print("\n=== Разделение по пользователям ===")

np.random.seed(42)

# Собираем уникальных пользователей (по первому файлу)
user_files = files_sample
n_users = len(user_files)
n_train = int(n_users * 0.8)

train_users = user_files[:n_train]
test_users = user_files[n_train:]

X_train, y_train = [], []
X_test, y_test = [], []

for file in train_users:
    df = pd.read_parquet(os.path.join(DATA_PATH, file))
    X, y, _ = create_sequences(df, source_to_idx, platform_to_idx, seq_len=10)
    if X is not None:
        X_train.append(X)
        y_train.append(y)

for file in test_users:
    df = pd.read_parquet(os.path.join(DATA_PATH, file))
    X, y, _ = create_sequences(df, source_to_idx, platform_to_idx, seq_len=10)
    if X is not None:
        X_test.append(X)
        y_test.append(y)

X_train = np.concatenate(X_train, axis=0)
y_train = np.concatenate(y_train, axis=0)
X_test = np.concatenate(X_test, axis=0)
y_test = np.concatenate(y_test, axis=0)

print(f"Train: {X_train.shape[0]} примеров")
print(f"Test: {X_test.shape[0]} примеров")


=== Разделение по пользователям ===
Train: 239335 примеров
Test: 43132 примеров


In [23]:
class TimeSeriesDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y).unsqueeze(1)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [25]:
train_dataset = TimeSeriesDataset(X_train, y_train)
test_dataset = TimeSeriesDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

In [19]:
class LSTMPredictor(nn.Module):
    """
    LSTM модель для предсказания следующего ответа
    """
    def __init__(self, input_dim=11, hidden_dim=128, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, dropout=dropout)
        self.layer_norm = nn.LayerNorm(hidden_dim)
        self.fc1 = nn.Linear(hidden_dim, 64)
        self.fc2 = nn.Linear(64, 1)
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        # LSTM обрабатывает последовательность
        lstm_out, (h_n, c_n) = self.lstm(x)
        
        # Берем последний скрытый state
        last_out = lstm_out[:, -1, :]
        
        # Полносвязные слои
        out = self.layer_norm(last_out)
        out = self.relu(self.fc1(out))
        out = self.dropout(out)
        out = self.sigmoid(self.fc2(out))
        
        return out

In [26]:
print("\n=== Обучение модели ===")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

model = LSTMPredictor(input_dim=11, hidden_dim=128, num_layers=2, dropout=0.2).to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=3)

best_auc = 0
best_model = None

for epoch in range(20):
    # Training
    model.train()
    train_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        pred = model(X_batch)
        loss = criterion(pred, y_batch)
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item()
    
    # Validation
    model.eval()
    all_preds, all_true = [], []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            pred = model(X_batch)
            all_preds.extend(pred.cpu().numpy().flatten())
            all_true.extend(y_batch.numpy().flatten())
    
    auc = roc_auc_score(all_true, all_preds)
    acc = accuracy_score(all_true, np.array(all_preds) > 0.5)
    
    scheduler.step(auc)
    
    if auc > best_auc:
        best_auc = auc
        best_model = model.state_dict().copy()
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1:2d}: loss={train_loss/len(train_loader):.4f}, AUC={auc:.4f}, ACC={acc:.4f}")


=== Обучение модели ===
Device: cuda
Epoch  5: loss=0.5935, AUC=0.6118, ACC=0.7118
Epoch 10: loss=0.5923, AUC=0.6140, ACC=0.7148
Epoch 15: loss=0.5906, AUC=0.6143, ACC=0.7147
Epoch 20: loss=0.5903, AUC=0.6146, ACC=0.7148


In [28]:
print("\n=== Финальная оценка ===")

model.load_state_dict(best_model)
model.eval()

all_preds, all_true = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        pred = model(X_batch)
        all_preds.extend(pred.cpu().numpy().flatten())
        all_true.extend(y_batch.numpy().flatten())

all_preds = np.array(all_preds)
all_true = np.array(all_true)
all_pred_binary = (all_preds > 0.5).astype(int)

print(f"\nРезультаты на тесте:")
print(f"  AUC: {roc_auc_score(all_true, all_preds):.4f}")
print(f"  Accuracy: {accuracy_score(all_true, all_pred_binary):.4f}")
print(f"  LogLoss: {log_loss(all_true, all_preds):.4f}")

print("\n=== Анализ важности признаков (через корреляцию) ===")

feature_names = [
    'attempt_count', 'time_to_answer', 'lectures_watched',
    'platform', 'source', 'hour', 'day_of_week',
    'rolling_accuracy', 'time_since_last', 'current_correct', 'prev_correct'
]

last_step = X_test[:, -1, :]
correlations = []
for i, name in enumerate(feature_names):
    corr = np.corrcoef(last_step[:, i], all_true)[0, 1]
    correlations.append((name, corr))

for name, corr in sorted(correlations, key=lambda x: abs(x[1]), reverse=True):
    print(f"  {name}: {corr:.4f}")


=== Финальная оценка ===

Результаты на тесте:
  AUC: 0.6146
  Accuracy: 0.7148
  LogLoss: 0.5859

=== Анализ важности признаков (через корреляцию) ===
  rolling_accuracy: 0.1770
  current_correct: 0.0928
  prev_correct: 0.0881
  source: -0.0498
  platform: 0.0168
  time_to_answer: 0.0161
  hour: -0.0128
  attempt_count: -0.0106
  lectures_watched: 0.0045
  day_of_week: 0.0028
  time_since_last: -0.0026
